## Ricavo funzione
In questo codice ricavo la funzione tramite il fit dei dati della misura con solo sorgente + primo polarizzatore.
Se la sorgente è parzialmente polarizzata: I1(θ) = (I0/2)[1 + p cos(2θ − 2φ0)], cioè periodo 180° (2θ), non 4θ.

In [1]:
import ROOT
import numpy as np
import pandas as pd
from scipy.stats import f as f_dist
 
ROOT.gStyle.SetOptFit(1111)    # aggiunge il label con chi2/ndf e probabilita e parametri se metti 0000 toglie i dati
ROOT.gStyle.SetFitFormat(
    "4.2e"
)  # con 2 cifre decimali

In [2]:
#fit
# luce sorgente dopo il primo polarizzatore (ruotato di un angolo θ):
#     V(θ) = A * (1 + E*cos(2θ - F)) * (1 + D*θ)
# E = grado di polarizzazione P della sorgente,  F = 2*φ0 (fase)
def Sorgente(x, par):
    t = x[0]
    A = par[0]
    E = par[1]
    F = par[2]
    D = par[3]
    return A*(1 + E*np.cos(2*t - F))*(1 + D*t)

def modellopy_sorgente(t, A, E, F, D):
    return A*(1 + E*np.cos(2*t - F))*(1 + D*t)

def Sorgente_deriv(x, par):
    t = x
    dt = 1e-6
    return ((modellopy_sorgente(t+dt/2, par[0], par[1], par[2], par[3])-modellopy_sorgente(t-dt/2, par[0], par[1], par[2], par[3]))/dt)

In [3]:
xlsx = "Esperienza4.xlsx"         
df = pd.read_excel(xlsx, sheet_name= "Dati")
df = df.dropna(subset=["θ [rad]", "V", "sigma_θ", "sigma_V"])
df = df.to_numpy(dtype = 'float64')

x = df[:, 1]     
y = df[:, 2]     

err_x = df[:, 3]     
err_y = df[:, 4]   

# eliminazione dati (0 = non scarto nulla)
N_SCARTA = 0
index = [i for i in range(N_SCARTA)]

x = np.delete(x, index)
y = np.delete(y, index)
err_x = np.delete(err_x, index)
err_y = np.delete(err_y, index)

In [4]:
#minimizzazzione con ROOT
SorgenteFit = ROOT.TF1('model', Sorgente, min(x), max(x), 4)
E0 = (y.max() - y.min())/(y.max() + y.min())                         #  modulazione: valore di partenza per E
SorgenteFit.SetParameters(y.mean(), E0, 2*x[np.argmax(y)], 0)        # A è la media, E la modulazione, F è 2*(posizione del massimo)
SorgenteFit.SetParNames("A", "E", "F", "D")

gr1 = ROOT.TGraphErrors(len(x), x, y, err_x, err_y)
gr1.SetTitle("Dati sperimentali e fit; #theta [rad]; V [V]")
gr1.SetMarkerStyle(20)
gr1.SetMarkerSize(0.7)                    
gr1.SetMarkerColor(ROOT.kAzure + 2)
gr1.SetLineColor(ROOT.kAzure + 3)        
gr1.SetLineWidth(1)


fit_res = gr1.Fit(SorgenteFit, "S") # SR significa che voglio fare il fit senza disegnare la funzione e senza stampare a video i risultati del fit 

fitStatus = fit_res.Status()
print(f"Fit status: {fitStatus}")

par = SorgenteFit.GetParameters()

errors = SorgenteFit.GetParErrors()

chi2 = fit_res.Chi2()
ndf = fit_res.Ndf()

err_yeff = np.sqrt(err_y**2 + ((Sorgente_deriv(x, par)) * err_x)**2) #errore effettivo

print("Parametri stimati:")
print(f"A = {par[0]} ± {errors[0]}")
print(f"E = {par[1]} ± {errors[1]}")
print(f"F = {par[2]} ± {errors[2]}")
print(f"D = {par[3]} ± {errors[3]}")
print(f"Chi2 = {chi2}, ndf = {ndf}")

# (E, F) e (-E, F+π) descrivono la stessa curva: scelgo E positivo e F tra 0 e 2π
E_new, errE_new = par[1], errors[1]
F_new, errF_new = par[2], errors[2]
if E_new < 0:
    E_new = -E_new
    F_new = F_new + np.pi
F_new = F_new % (2*np.pi)
print(f"P = E = {E_new*100:.2f} ± {errE_new*100:.2f} %")
print(f"F = {F_new:.3f} ± {errF_new:.3f} rad = {np.degrees(F_new):.1f}°")

c1 = ROOT.TCanvas("c1", "Fit con ROOT", 800, 600)
c1.SetGrid()

gr1.Draw("AP")


leg1 = ROOT.TLegend(0.14, 0.14, 0.42, 0.23)
leg1.AddEntry(gr1, "Dati", "pe")
leg1.AddEntry(SorgenteFit, "Fit", "l")
leg1.Draw()

c1.Draw()
c1.SaveAs("Immagini/sorgente_fit.png")

****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      659.665
NDf                       =          196
Edm                       =  2.67157e-06
NCalls                    =          101
A                         =      3.79962   +/-   0.00219623  
E                         =     0.262657   +/-   0.000397035 
F                         =   -0.0436931   +/-   0.00206028  
D                         =  -0.00169565   +/-   0.00013761  
Fit status: 0
Parametri stimati:
A = 3.7996196849742416 ± 0.0021962334561549013
E = 0.2626574337750465 ± 0.0003970353775420326
F = -0.04369307057588286 ± 0.002060275558675634
D = -0.0016956505527153374 ± 0.00013760958898322245
Chi2 = 659.6652685775384, ndf = 196
P = E = 26.27 ± 0.04 %
F = 6.239 ± 0.002 rad = 357.5°


Info in <TCanvas::Print>: png file Immagini/sorgente_fit.png has been created


## Confronto
Confronto E ed F ricavati ora (solo sorgente + P1) con quelli ricavati dalla misura completa di Malus, dove la sorgente veniva descritta dal fattore (1 + E cos(2θ - F)). Se la causa è la polarizzazione parziale della sorgente i due risultati devono essere compatibili.

Il confronto di F ha senso solo se la scala del polarizzatore (lo zero) e la posizione/orientazione della lampada non sono cambiate rispetto alla misura di prima.

In [ ]:
# valori ottenuti dal fit della misura di Malus
E_old, errE_old = 0.0673, 0.0024
F_old, errF_old = 2.777, 0.036            # rad

# differenza in unità di sigma
dE = (E_new - E_old) / np.sqrt(errE_new**2 + errE_old**2)
dF = ((F_new - F_old + np.pi) % (2*np.pi) - np.pi) / np.sqrt(errF_new**2 + errF_old**2)   # differenza di F riportata tra -π e π

print(f"E: ora = {E_new*100:.2f} ± {errE_new*100:.2f} %   prima = {E_old*100:.2f} ± {errE_old*100:.2f} %   differenza = {dE:.1f} sigma")
print(f"F: ora = {F_new:.3f} ± {errF_new:.3f} rad   prima = {F_old:.3f} ± {errF_old:.3f} rad   differenza = {dF:.1f} sigma")

if abs(dE) < 2 and abs(dF) < 2:
    print("Compatibili entro 2 sigma: la polarizzazione della sorgente spiega l'onda trovata.")
else:
    print("NON compatibili: la polarizzazione della sorgente non basta a spiegare l'onda.")

E: ora = 26.27 ± 0.04 %   prima = 6.73 ± 0.24 %   differenza = 80.3 sigma
F: ora = 6.239 ± 0.002 rad   prima = 2.777 ± 0.036 rad   differenza = -78.2 sigma
NON compatibili: la polarizzazione della sorgente non basta a spiegare l'onda.
